In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import json
import plotly.express as px
import plotly.graph_objects as go
from utils.hilfsfunktionen import ( 
    select_dataframe, 
    find_relevant_years, 
    short_analysis_frame, 
    dump_json
)
from src.preparations import load_config
from src.analysis import create_dev_change_columns, create_indicator_frames
from src.paths import (
    EDUCATION_RAW,
    DEVELOPMENT_CONFIG,
    EDUCATION_CONFIG,
    EDUCATION_OUTPUT,
    DEVELOPMENT_OUTPUT,
    COUNTRY_INFO
)

df_edu_raw = pd.read_csv(EDUCATION_RAW, encoding="utf-8")

In [3]:
df_edu_raw.head(10)

,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
0,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.F,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.M,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arab World,ARB,"Adjusted net enrolment rate, primary, both sex...",SE.PRM.TENR,54.822121,54.894138,56.209438,57.267109,57.991138,59.365540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Arab World,ARB,"Adjusted net enrolment rate, primary, female (%)",SE.PRM.TENR.FE,43.351101,43.318150,44.640701,45.845718,46.449501,48.363892,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Arab World,ARB,"Adjusted net enrolment rate, primary, gender p...",UIS.NERA.1.GPI,0.658570,0.656400,0.663290,0.672040,0.672610,0.691760,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Arab World,ARB,"Adjusted net enrolment rate, primary, male (%)",SE.PRM.TENR.MA,65.826233,65.993584,67.301857,68.219078,69.059013,69.914551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Arab World,ARB,"Adjusted net enrolment rate, upper secondary, ...",UIS.NERA.3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Arab World,ARB,"Adjusted net enrolment rate, upper secondary, ...",UIS.NERA.3.F,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_edu_raw.shape

(886930, 70)

In [5]:
df_edu_raw.head(10)

,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
0,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.F,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.M,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arab World,ARB,"Adjusted net enrolment rate, primary, both sex...",SE.PRM.TENR,54.822121,54.894138,56.209438,57.267109,57.991138,59.365540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Arab World,ARB,"Adjusted net enrolment rate, primary, female (%)",SE.PRM.TENR.FE,43.351101,43.318150,44.640701,45.845718,46.449501,48.363892,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Arab World,ARB,"Adjusted net enrolment rate, primary, gender p...",UIS.NERA.1.GPI,0.658570,0.656400,0.663290,0.672040,0.672610,0.691760,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Arab World,ARB,"Adjusted net enrolment rate, primary, male (%)",SE.PRM.TENR.MA,65.826233,65.993584,67.301857,68.219078,69.059013,69.914551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Arab World,ARB,"Adjusted net enrolment rate, upper secondary, ...",UIS.NERA.3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Arab World,ARB,"Adjusted net enrolment rate, upper secondary, ...",UIS.NERA.3.F,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
indicators = list(df_edu_raw["Indicator Name"].unique())
indicators

['Adjusted net enrolment rate, lower secondary, both sexes (%)',
 'Adjusted net enrolment rate, lower secondary, female (%)',
 'Adjusted net enrolment rate, lower secondary, gender parity index (GPI)',
 'Adjusted net enrolment rate, lower secondary, male (%)',
 'Adjusted net enrolment rate, primary, both sexes (%)',
 'Adjusted net enrolment rate, primary, female (%)',
 'Adjusted net enrolment rate, primary, gender parity index (GPI)',
 'Adjusted net enrolment rate, primary, male (%)',
 'Adjusted net enrolment rate, upper secondary, both sexes (%)',
 'Adjusted net enrolment rate, upper secondary, female (%)',
 'Adjusted net enrolment rate, upper secondary, gender parity index (GPI)',
 'Adjusted net enrolment rate, upper secondary, male (%)',
 'Adjusted net intake rate to Grade 1 of primary education, both sexes (%)',
 'Adjusted net intake rate to Grade 1 of primary education, female (%)',
 'Adjusted net intake rate to Grade 1 of primary education, gender parity index (GPI)',
 'Adjusted 

In [7]:
indicators_group = []
for ind in indicators:
    parts = ind.split(",")
    first_part = parts[0]
    if first_part not in indicators_group:
        indicators_group.append(first_part)

indicators_group

['Adjusted net enrolment rate',
 'Adjusted net intake rate to Grade 1 of primary education',
 'Adult illiterate population',
 'Adult literacy rate',
 'Africa Dataset: Average number of grades per multigrade class in primary schools (number of grades)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 1 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 2 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 3 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 4 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 5 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 6 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 7 of

In [8]:
indicators_group2 = []
for ind in indicators_group:
    parts = ind.split(".")
    first_part = parts[0]
    if first_part not in indicators_group2:
        indicators_group2.append(first_part)

indicators_group2

['Adjusted net enrolment rate',
 'Adjusted net intake rate to Grade 1 of primary education',
 'Adult illiterate population',
 'Adult literacy rate',
 'Africa Dataset: Average number of grades per multigrade class in primary schools (number of grades)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 1 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 2 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 3 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 4 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 5 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 6 of primary education (number)',
 'Africa Dataset: Average number of pupils per mathematics textbook in Grade 7 of

In [9]:
indicators_group3 = []
for ind in indicators_group2:
    parts = ind.split(":")
    first_part = parts[0]
    if first_part not in indicators_group3:
        indicators_group3.append(first_part)

indicators_group3

['Adjusted net enrolment rate',
 'Adjusted net intake rate to Grade 1 of primary education',
 'Adult illiterate population',
 'Adult literacy rate',
 'Africa Dataset',
 'All staff compensation as % of total expenditure in lower secondary public institutions (%)',
 'All staff compensation as % of total expenditure in post-secondary non-tertiary public institutions (%)',
 'All staff compensation as % of total expenditure in pre-primary public institutions (%)',
 'All staff compensation as % of total expenditure in primary public institutions (%)',
 'All staff compensation as % of total expenditure in public institutions (%)',
 'All staff compensation as % of total expenditure in secondary public institutions (%)',
 'All staff compensation as % of total expenditure in tertiary public institutions (%)',
 'All staff compensation as % of total expenditure in upper secondary public institutions (%)',
 'Annual statutory teacher salaries in public institutions in USD',
 'Barro-Lee',
 'Capital e

## Erste Übersicht bzgl. Datenqualität und -struktur

In [10]:
short_analysis_frame(df_edu_raw)


========== DataFrame Qualität ==========


Anzahl Zeilen: 886930
davon: 0 Duplikate

Anzahl Spalten: 70
davon:
    66 numerisch
    0 datetime
    4 string/object
    0 kategorial

               Datentyp  Fehlende Werte  Eindeutige Werte
Country Name        str            0.00               242
Country Code        str            0.00               242
Indicator Name      str            0.00              3665
Indicator Code      str            0.00              3665
1970            float64           91.85             24595
...                 ...             ...               ...
2085            float64           94.20              7335
2090            float64           94.20              7150
2095            float64           94.20              7044
2100            float64           94.20              6914
Unnamed: 69     float64          100.00                 0

[70 rows x 3 columns]


## Entwicklung einer Funktion zum Auffinden für den Indikator relevanter Jahre

In [11]:
EDU_INDICATOR_CODE = "LO.TIMSS.SCI8"
DEV_INDICATOR_CODE = "NY.GDP.PCAP.KD"
TIMSS_YEARS_LIST = ["1999", "2003", "2007", "2011", "2015", "2019", "2023"]

filter_timss = df_edu_raw["Indicator Code"] == EDU_INDICATOR_CODE
df_indicator = df_edu_raw[filter_timss]
records_total = df_indicator.shape[0]

col_list = df_indicator.columns

year_cols = []

for col in col_list:
    if col.isdigit():
        year_cols.append(col)

year_cols_relevant = []

for col in year_cols:
    values_count = df_indicator[col].notna().sum()
    if values_count > 0:
        year_cols_relevant.append(col)

year_cols_relevant

['1995', '1999', '2003', '2007', '2011', '2015']

In [12]:
years = find_relevant_years(df_indicator)
years

['1995', '1999', '2003', '2007', '2011', '2015']

## Extrahieren relevanter Bildungsdaten

- die internationalen Bildungsdaten werden auf den relevanten TIMSS-Score (Science) gefiltert
- danach wird nur auf Score-relevante Jahre gefiltert

In [13]:
df_base = select_dataframe("edu")

df_timss_science, df_timss_science_long = create_indicator_frames(frame_name="edu", indicator_code=EDU_INDICATOR_CODE, indicator_title="timss_science")

df_timss_science

,country_name,country_code,indicator_name,indicator_code,timss_science_data_count,1995,1999,2003,2007,2011,2015
85,Algeria,DZA,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,1,NaN,NaN,NaN,408.059523,NaN,NaN
230,Armenia,ARM,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,3,NaN,NaN,461.266680,487.960381,436.924652,NaN
296,Australia,AUS,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,6,514.000000,540.258156,527.000000,515.000000,519.000000,512.0
335,Austria,AUT,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,1,538.933284,NaN,NaN,NaN,NaN,NaN
415,Bahrain,BHR,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,4,NaN,NaN,438.000000,467.000000,452.000000,466.0
...,...,...,...,...,...,...,...,...,...,...,...
5573,Turkey,TUR,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,4,NaN,432.950957,NaN,454.158975,483.000000,493.0
5672,Ukraine,UKR,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,2,NaN,NaN,NaN,485.062954,500.991769,NaN
5706,United Arab Emirates,ARE,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,2,NaN,NaN,NaN,NaN,465.000000,477.0
5777,United States,USA,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,6,513.000000,515.000000,527.000000,520.000000,525.000000,530.0


## Nutzbar machen der Scoredaten
- der DataFrame wird ins Long-Format gebracht. 
- Zeilen ohne Score werden direkt als irrelevant entfernt 

In [14]:
df_timss_science_long.columns = (
        df_timss_science_long.columns
        .str.replace("_", " ")
        .str.title()
    )

years = find_relevant_years(df_timss_science)

for year in years:
    df_year = df_timss_science_long[df_timss_science_long["Year"] == int(year)]
    
    fig = px.bar(
        df_year.sort_values("Timss Science Score").tail(10),
        x="Timss Science Score",
        y="Country Name",
        title=f"TIMSS Science: Top 20 Countries ({year})",
    )
    # fig.show()

## Datenstruktur der zwei neu erzeugten Tabellen

In [15]:
df_edu = pd.read_csv(EDUCATION_OUTPUT, encoding="utf-8")
df_dev = pd.read_csv(DEVELOPMENT_OUTPUT, encoding="utf-8")

short_analysis_frame(df_edu)


========== DataFrame Qualität ==========


Anzahl Zeilen: 6029
davon: 0 Duplikate

Anzahl Spalten: 51
davon:
    47 numerisch
    0 datetime
    4 string/object
    0 kategorial

               Datentyp  Fehlende Werte  Eindeutige Werte
country_name        str            0.00               209
country_code        str            0.00               209
indicator_name      str            0.00                46
indicator_code      str            0.00                46
1970            float64           93.91               362
1971            float64           79.37              1226
1972            float64           80.94              1142
1973            float64           80.74              1157
1974            float64           81.19              1133
1975            float64           80.94              1146
1976            float64           79.90              1206
1977            float64           80.10              1196
1978            float64           80.18              1191
1979    

In [16]:
short_analysis_frame(df_dev)


========== DataFrame Qualität ==========


Anzahl Zeilen: 4951
davon: 0 Duplikate

Anzahl Spalten: 30
davon:
    26 numerisch
    0 datetime
    4 string/object
    0 kategorial

               Datentyp  Fehlende Werte  Eindeutige Werte
country_name        str            0.00               214
country_code        str            0.00               214
indicator_name      str            0.00                26
indicator_code      str            0.00                26
1999            float64           35.93              2680
2000            float64           26.56              3108
2001            float64           33.91              2808
2002            float64           24.92              3197
2003            float64           24.92              3173
2004            float64           21.33              3330
2005            float64           21.47              3324
2006            float64           18.58              3459
2007            float64           16.60              3545
2008    

In [17]:
df_dev

,country_name,country_code,indicator_name,indicator_code,1999,2000,2001,2002,2003,2004,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AFG,Exports of goods and services (% of GDP),NE.EXP.GNFS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10.420817,14.342153,18.380042,16.235278,15.702752
1,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,NaN,308.318270,277.118051,338.139974,346.071627,338.637274,...,565.569730,563.872337,562.769574,553.125152,557.861533,527.834554,408.625855,377.665627,378.066303,374.376696
2,Afghanistan,AFG,Government Effectiveness - Governance estimate...,GOV_WGI_GE_EST,NaN,-2.344602,NaN,-2.098876,-2.108121,-1.156126,...,-1.471172,-1.417730,-1.442187,-1.631250,-1.654080,-1.716728,-1.705904,-1.964611,-2.092459,-1.933997
3,Afghanistan,AFG,High-technology exports (% of manufactured exp...,TX.VAL.TECH.MF.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,Imports of goods and services (% of GDP),NE.IMP.GNFS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,36.289077,37.069564,54.505427,50.764300,68.060273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4946,Zimbabwe,ZWE,"Prevalence of HIV, total (% of population ages...",SH.DYN.AIDS.ZS,27.000000,25.800000,24.500000,23.200000,21.900000,20.600000,...,14.500000,14.100000,13.600000,13.100000,12.500000,12.000000,11.500000,10.900000,10.300000,9.800000
4947,Zimbabwe,ZWE,Renewable energy consumption (% of total final...,EG.FEC.RNEW.ZS,64.400000,69.500000,72.300000,74.700000,78.000000,81.800000,...,80.800000,81.700000,82.000000,79.700000,81.000000,84.100000,82.400000,NaN,NaN,NaN
4948,Zimbabwe,ZWE,Total greenhouse gas emissions excluding LULUC...,EN.GHG.ALL.PC.CE.AR5,2.919746,2.627267,2.623309,2.373381,2.079466,1.989973,...,1.977177,1.821388,1.758746,1.864458,1.733276,1.596830,1.718639,1.744073,1.843543,1.850644
4949,Zimbabwe,ZWE,"Trademark applications, resident, by count",IP.TMK.RSCT,NaN,NaN,NaN,NaN,NaN,NaN,...,285.000000,155.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df_timss, df_timss_long = create_indicator_frames(indicator_code=EDU_INDICATOR_CODE, frame_name="edu", indicator_title="timss_science")
df_gdp_per_capita, df_gdp_per_capita_long = create_indicator_frames(indicator_code=DEV_INDICATOR_CODE, frame_name="dev", indicator_title="gdp_per_capita")

df_timss.head(10)

,country_name,country_code,indicator_name,indicator_code,timss_science_data_count,1995,1999,2003,2007,2011,2015
85,Algeria,DZA,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,1,NaN,NaN,NaN,408.059523,NaN,NaN
230,Armenia,ARM,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,3,NaN,NaN,461.266680,487.960381,436.924652,NaN
296,Australia,AUS,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,6,514.000000,540.258156,527.000000,515.000000,519.000000,512.0
335,Austria,AUT,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,1,538.933284,NaN,NaN,NaN,NaN,NaN
415,Bahrain,BHR,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,4,NaN,NaN,438.000000,467.000000,452.000000,466.0
701,Bosnia and Herzegovina,BIH,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,1,NaN,NaN,NaN,465.745173,NaN,NaN
739,Botswana,BWA,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,2,NaN,NaN,364.569248,354.534086,NaN,NaN
844,Bulgaria,BGR,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,3,NaN,518.010990,478.843142,470.283975,NaN,NaN
1029,Canada,CAN,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,3,513.987704,533.081997,NaN,NaN,NaN,526.0
1131,Chile,CHL,TIMSS: Mean performance on the science scale f...,LO.TIMSS.SCI8,4,NaN,420.000000,413.000000,NaN,461.000000,454.0


In [19]:
short_analysis_frame(df_gdp_per_capita)


========== DataFrame Qualität ==========


Anzahl Zeilen: 212
davon: 0 Duplikate

Anzahl Spalten: 31
davon:
    27 numerisch
    0 datetime
    4 string/object
    0 kategorial

                          Datentyp  Fehlende Werte  Eindeutige Werte
country_name                   str            0.00               212
country_code                   str            0.00               212
indicator_name                 str            0.00                 1
indicator_code                 str            0.00                 1
gdp_per_capita_data_count    int64            0.00                14
1999                       float64            7.55               196
2000                       float64            6.60               198
2001                       float64            6.60               198
2002                       float64            4.72               202
2003                       float64            4.72               202
2004                       float64            4.72            

In [20]:
df_edu[df_edu["1994"].notna()]

,country_name,country_code,indicator_name,indicator_code,1970,1971,1972,1973,1974,1975,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016
8,Afghanistan,AFG,"Gross enrolment ratio, secondary, both sexes (%)",SE.SEC.ENRR,8.331610,9.350290,10.348610,10.83169,10.976400,11.04103,...,30.083160,40.223381,46.732761,53.246830,54.616180,56.677341,56.688660,55.656158,55.644409,NaN
15,Afghanistan,AFG,Pupil-teacher ratio in secondary education (he...,SE.SEC.ENRL.TC.ZS,23.137621,22.521250,21.701401,20.51185,21.967581,NaN,...,31.562361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.720879,NaN
18,Afghanistan,AFG,"School life expectancy, primary and secondary,...",UIS.SLE.123,2.380400,2.493640,2.571360,2.62233,2.759030,2.66144,...,8.261090,8.800630,8.937480,9.526070,9.445550,9.936350,10.095130,10.101560,NaN,NaN
20,Afghanistan,AFG,"School life expectancy, secondary, both sexes ...",UIS.SLE.23,0.499900,0.561020,0.620920,0.64990,0.687560,0.66246,...,1.794800,2.413400,2.803970,3.194810,3.276970,3.400640,3.478320,3.384140,NaN,NaN
25,Albania,ALB,"Enrolment in tertiary education per 100,000 in...",UIS.TE_100000.56,NaN,1167.769775,1283.782227,NaN,NaN,NaN,...,2885.000244,3052.736084,3178.929199,4215.400879,4673.476562,5583.394531,6001.114746,6015.172852,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5996,Zambia,ZMB,"School life expectancy, secondary, both sexes ...",UIS.SLE.23,0.647350,0.670670,0.694140,0.67625,0.703310,0.74178,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6005,Zimbabwe,ZWE,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.973330,NaN,8.383220,8.485360,8.429330,NaN,NaN
6008,Zimbabwe,ZWE,"Gross enrolment ratio, secondary, both sexes (%)",SE.SEC.ENRR,7.381360,7.567980,8.159130,8.37640,8.422430,8.68479,...,NaN,NaN,NaN,NaN,NaN,46.673180,47.570190,NaN,NaN,NaN
6022,Zimbabwe,ZWE,Pupil-teacher ratio in secondary education (he...,SE.SEC.ENRL.TC.ZS,NaN,NaN,NaN,19.59516,19.379539,NaN,...,NaN,NaN,NaN,NaN,NaN,22.431910,22.483530,NaN,NaN,NaN


In [21]:
edu_config = load_config(EDUCATION_CONFIG)
indicators = edu_config["indicators"]
anzahl_indikatoren = len(indicators)

count_20 = 0
count_10 = 0

for indicator in indicators:
    values = indicators[indicator]
    education_years = values["education_years"]
    if education_years["10"]:
        count_10 += 1
    if education_years["20"]:
        count_20 += 1

print(f"20 Jahre: {count_20} / {anzahl_indikatoren}")
print(f"10 Jahre: {count_10} / {anzahl_indikatoren}")

20 Jahre: 29 / 46
10 Jahre: 34 / 46


In [22]:
short_analysis_frame(df_dev)
df_dev["indicator_name"].unique()


========== DataFrame Qualität ==========


Anzahl Zeilen: 4951
davon: 0 Duplikate

Anzahl Spalten: 30
davon:
    26 numerisch
    0 datetime
    4 string/object
    0 kategorial

               Datentyp  Fehlende Werte  Eindeutige Werte
country_name        str            0.00               214
country_code        str            0.00               214
indicator_name      str            0.00                26
indicator_code      str            0.00                26
1999            float64           35.93              2680
2000            float64           26.56              3108
2001            float64           33.91              2808
2002            float64           24.92              3197
2003            float64           24.92              3173
2004            float64           21.33              3330
2005            float64           21.47              3324
2006            float64           18.58              3459
2007            float64           16.60              3545
2008    

<ArrowStringArray>
[                                                       'Exports of goods and services (% of GDP)',
                                                              'GDP per capita (constant 2015 US$)',
                           'Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)',
                                             'High-technology exports (% of manufactured exports)',
                                                        'Imports of goods and services (% of GDP)',
                                                'Individuals using the Internet (% of population)',
                                       'Industry (including construction), value added (% of GDP)',
                                                      'Intentional homicides (per 100,000 people)',
 'Labor force with advanced education (% of total working-age population with advanced education)',
                                                         'Life expectancy at birt

In [23]:
df_dev.head()

,country_name,country_code,indicator_name,indicator_code,1999,2000,2001,2002,2003,2004,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AFG,Exports of goods and services (% of GDP),NE.EXP.GNFS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10.420817,14.342153,18.380042,16.235278,15.702752
1,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,NaN,308.318270,277.118051,338.139974,346.071627,338.637274,...,565.569730,563.872337,562.769574,553.125152,557.861533,527.834554,408.625855,377.665627,378.066303,374.376696
2,Afghanistan,AFG,Government Effectiveness - Governance estimate...,GOV_WGI_GE_EST,NaN,-2.344602,NaN,-2.098876,-2.108121,-1.156126,...,-1.471172,-1.417730,-1.442187,-1.631250,-1.654080,-1.716728,-1.705904,-1.964611,-2.092459,-1.933997
3,Afghanistan,AFG,High-technology exports (% of manufactured exp...,TX.VAL.TECH.MF.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,Imports of goods and services (% of GDP),NE.IMP.GNFS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,36.289077,37.069564,54.505427,50.764300,68.060273


In [24]:
year_columns = [
    col 
    for col in df_dev.columns 
    if col.isdigit()
]

df_zero = df_dev[(df_dev[year_columns] == 0).any(axis=1)]

df_zero

,country_name,country_code,indicator_name,indicator_code,1999,2000,2001,2002,2003,2004,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
3,Afghanistan,AFG,High-technology exports (% of manufactured exp...,TX.VAL.TECH.MF.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
82,American Samoa,ASM,Renewable energy consumption (% of total final...,EG.FEC.RNEW.ZS,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.200000,0.200000,0.300000,0.300000,0.400000,0.400000,0.400000,0.400000,NaN,NaN
129,Antigua and Barbuda,ATG,High-technology exports (% of manufactured exp...,TX.VAL.TECH.MF.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.026056,0.375050,0.613427,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
145,Antigua and Barbuda,ATG,Renewable energy consumption (% of total final...,EG.FEC.RNEW.ZS,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.400000,0.400000,0.400000,0.700000,0.700000,0.900000,0.900000,0.900000,NaN,NaN
292,"Bahamas, The",BHS,High-technology exports (% of manufactured exp...,TX.VAL.TECH.MF.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,2.104437,7.802953,1.966442,1.205449,3.044763,0.261200,0.000000,0.013825,0.000000
307,"Bahamas, The",BHS,Renewable energy consumption (% of total final...,EG.FEC.RNEW.ZS,0.000000,0.000000,0.400000,0.400000,1.5,1.600000,...,1.400000,1.200000,1.100000,1.100000,1.100000,1.400000,1.100000,1.100000,NaN,NaN
329,Bahrain,BHR,Renewable energy consumption (% of total final...,EG.FEC.RNEW.ZS,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
436,Belize,BLZ,High-technology exports (% of manufactured exp...,TX.VAL.TECH.MF.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000082,0.066546,0.000000,0.938241,0.191640,0.000182,0.000000,0.000191
642,Brunei Darussalam,BRN,Renewable energy consumption (% of total final...,EG.FEC.RNEW.ZS,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
725,Cabo Verde,CPV,High-technology exports (% of manufactured exp...,TX.VAL.TECH.MF.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.000096,1.323279,0.760205,2.038021,3.476540,1.152150,NaN


In [25]:
df_dev = pd.read_csv(DEVELOPMENT_OUTPUT, encoding="utf-8")

dev_config = load_config(DEVELOPMENT_CONFIG)

df_dev_armut = df_dev[df_dev["indicator_code"] == "VC.IHR.PSRC.P5"]

df_dev_armut = create_dev_change_columns(df_dev_armut, dev_config)

df_dev_armut.sort_values("country_name")

short_analysis_frame(df_dev_armut)


========== DataFrame Qualität ==========


Anzahl Zeilen: 193
davon: 0 Duplikate

Anzahl Spalten: 32
davon:
    28 numerisch
    0 datetime
    4 string/object
    0 kategorial

                     Datentyp  Fehlende Werte  Eindeutige Werte
country_name              str            0.00               193
country_code              str            0.00               193
indicator_name            str            0.00                 1
indicator_code            str            0.00                 1
1999                  float64           46.11               104
2000                  float64           39.38               117
2001                  float64           34.20               127
2002                  float64           33.16               129
2003                  float64           43.01               110
2004                  float64           34.72               126
2005                  float64           36.27               123
2006                  float64           32.64        

In [26]:
df_dev_armut.sort_values("change_over_10_years").head(20)

,country_name,country_code,indicator_name,indicator_code,1999,2000,2001,2002,2003,2004,...,2017,2018,2019,2020,2021,2022,2023,2024,change_over_10_years,change_over_20_years
4797,"Venezuela, RB",VEN,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,24.822790,32.707202,31.865593,37.809107,43.803846,36.897080,...,47.979208,NaN,41.032413,29.475380,19.279105,12.646645,NaN,NaN,-41.735484,-25.162462
1914,Honduras,HND,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,40.043978,48.283823,51.612957,52.213199,57.211380,49.856182,...,40.299802,37.981772,41.011166,35.702849,38.202597,34.967935,31.442432,NaN,-40.903155,-25.768948
1350,El Salvador,SLV,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,65.016312,59.747288,60.153551,47.355520,55.946401,64.899476,...,63.764040,53.795877,38.538696,21.508745,17.343956,7.897688,NaN,NaN,-34.529702,-39.457832
440,Belize,BLZ,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,NaN,17.026649,25.767126,33.983247,25.399955,29.077418,...,38.049150,37.609162,34.730457,26.099539,31.617834,28.058292,NaN,NaN,-15.301137,-5.924955
3599,Puerto Rico (US),PRI,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,15.627981,19.265769,21.033235,21.518286,21.492138,22.192057,...,21.590371,20.181954,19.251661,16.972927,19.426966,17.649045,14.589658,NaN,-11.028839,-6.902479
1792,Guatemala,GTM,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,23.242652,24.825622,26.985430,29.648282,33.826071,35.190587,...,32.805474,28.903544,25.941600,18.188286,20.245871,22.047440,23.365727,NaN,-10.135846,-10.460344
606,Brazil,BRA,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,25.002106,26.074847,27.200608,27.848196,28.271088,26.491836,...,31.161176,27.160615,21.244560,21.812909,20.504863,20.081175,19.275296,NaN,-9.365115,-8.995793
946,Colombia,COL,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,63.333992,67.892159,70.109587,70.396095,57.460863,48.648188,...,25.551059,23.820759,23.399061,22.642703,25.580127,24.948808,24.913442,NaN,-8.502529,-32.547421
1273,Dominican Republic,DOM,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,12.621048,14.095681,12.556931,14.448329,21.178939,24.591176,...,11.399812,9.905349,9.417991,8.729777,10.536275,12.367848,10.916698,NaN,-8.487192,-10.262241
3420,Panama,PAN,"Intentional homicides (per 100,000 people)",VC.IHR.PSRC.P5,9.655054,9.932339,9.968932,12.015298,10.564903,9.473558,...,9.222421,9.621243,11.334922,11.646159,12.657048,11.338917,11.707293,NaN,-5.624547,1.142391
